In [0]:
# Bootstrap work

import sys
import os

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()

repo_root = "/Workspace" + "/".join(notebook_path.split("/")[:-2])

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"repo root: {repo_root}")

In [0]:
# Imports Cell

import yaml
from pyspark.sql import functions as F
from delta.tables import DeltaTable

from src.transformations.silver_transformer import (
    filter_quarantined,
    deduplicate,
    enrich_columns,
    silver_metadata_columns,
    select_silver_columns
) 

from src.utils.spark_session import get_spark_session

spark = get_spark_session(app_name="SilverTransformations")

print("Imports successful")

In [0]:
# Load configs

config_path = os.path.join(repo_root, "config", "pipeline_config.yml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

env = "dev"
cfg = config["environments"][env]
dq_cfg = config["data_quality"]

BRONZE_PATH = cfg["bronze_path"]
SILVER_PATH = cfg["silver_path"]

PIPELINE_RUN_ID = "silver_manual_run_001"

print(f"Environment: {env}")
print(f"Bronze path: {BRONZE_PATH}")
print(f"Silver path: {SILVER_PATH}")

In [0]:
# ── Cell 4: ADLS OAuth configuration ────────────────────
# Required in every notebook — Spark config does not
# persist between separate notebook sessions

from src.utils.spark_session import configure_adls_oauth

client_id     = dbutils.secrets.get(scope="kv-bank-etl-scope",
                                     key="databricks-sp-client-id")
client_secret = dbutils.secrets.get(scope="kv-bank-etl-scope",
                                     key="databricks-sp-client-secret")
tenant_id     = dbutils.secrets.get(scope="kv-bank-etl-scope",
                                     key="databricks-sp-tenant-id")

storage_account = cfg["storage_account"]

configure_adls_oauth(spark, storage_account, client_id, client_secret, tenant_id)

print("ADLS OAuth configured successfully")
print(f"Storage account : {storage_account}")

In [0]:
# Read from bronze delta table

df_bronze = spark.read.format("delta").load(BRONZE_PATH)

total = df_bronze.count()
quarantined = df_bronze.filter(F.col("_is_quarantined") == True).count()
clean = total - quarantined

print(f"Total Bronze Records: {total:,}")
print(f"Quarantined records: {quarantined:,}")
print(f"Clean records for Silver: {clean:,}")

In [0]:
# Apply silver transformations

df_clean = filter_quarantined(df_bronze)

df_deduped = deduplicate(
    df_clean, 
    partition_cols=["Amount", "Time", "Class"], 
    order_col="_ingestion_timestamp")

df_enriched = enrich_columns(df_deduped)

df_silver = silver_metadata_columns(df_enriched, PIPELINE_RUN_ID)

df_final = select_silver_columns(df_silver)

print(f"Final silver shape: {(df_final.count(), len(df_final.columns))}")
df_final.printSchema()

In [0]:
## Write to Silver Delta table using MERGE
# MERGE gives us exactly-once semantics:
# - If transaction_id already exists → UPDATE it
# - If new → INSERT it
# This makes the pipeline safely re-runnable (idempotent)

if DeltaTable.isDeltaTable(spark, SILVER_PATH):
    print("Silver Delta Table is already exist, hence running the merge logic")
    silver_table = DeltaTable.forPath(spark, SILVER_PATH)

    silver_table.alias("target").merge(
        df_final.alias("source"),
        "target.transaction_id = source.transaction_id"
        ).whenMatchedUpdateAll() \
            .whenNotMatchedInsertAll() \
                .execute()
    print("Merge operation completed successfully")

else:
    print("Silver Delta Table is not exist, hence writing the data to delta table")
    df_final \
        .write \
            .format("delta") \
                .mode("overwrite") \
                    .partitionBy("_ingestion_date") \
                        .save(SILVER_PATH)
    print("Initial Silver table created successfully")

# Verify
df_verify = spark.read.format("delta").load(SILVER_PATH)
print(f"Row count of Silver Table: {df_verify.count():,}")


In [0]:
# Spark Table Summary

print("=" * 10, "SILVER TABLE SUMMARY", "=" * 10)

df_silver = spark.read.format("delta").load(SILVER_PATH)

# Check if any quarantined records leaked through

df_filter = df_silver.filter(F.col("_is_quarantined") == True)
print(f"Number of Quarantined records in Silver: {df_filter.count():,}")

# Fraud distribution
print("\nFraud distribution:")
df_silver.groupBy("is_fraud") \
    .count() \
        .show()

# Amount bucket distribution
print("Amount bucket distribution:")
df_silver.groupBy("amount_bucket") \
    .count() \
        .orderBy("count", ascending=False) \
            .show()

# Delta history
print("\nSilver Delta table history:")
display(spark.sql(f"DESCRIBE HISTORY delta.`{SILVER_PATH}`"))